similarly, answer the below design question as well. Try to keep the framework same for answering.
Design a simple internal file upload service for profile pictures.

Employees should be able to:
upload a profile picture
retrieve/display their current profile picture
replace it later with a new one



Designing a simple internal file upload service for profile pictures requires a focus on **reliability** to prevent data loss and **low latency** for a smooth employee experience.

### **1. Requirements**
*   **Functional:** Employees must be able to **upload**, **retrieve**, and **replace** their profile pictures [User prompt].
*   **Non-Functional:**
    *   **Reliability:** Once a photo is uploaded, it must not be lost or corrupted.
    *   **Availability:** The service should remain accessible even if parts of the system fail.
    *   **Low Latency:** Retrieving a profile picture should be near-instant (ideally <100ms) to ensure fast page loads across internal tools.

### **2. API Sketch**
The service will use a RESTful approach:
*   **POST** `/v1/profile/upload`
    *   **Input:** Multi-part file data (the image).
    *   **Output:** `{ "photo_url": "string", "upload_status": "success" }`.
*   **GET** `/v1/profile/{user_id}`
    *   **Output:** The actual image file or a **redirect URL** to the storage location.
*   **PUT/POST** `/v1/profile/update`
    *   **Action:** Replaces the existing file entry with a new one and updates metadata.

### **3. Data Model**
The design uses a **decoupled approach** separating file content from descriptive data.
*   **Metadata DB (Relational):** A SQL database (like PostgreSQL) stores information such as `user_id`, `photo_url`, `file_size`, and `last_updated_at` to maintain **ACID consistency**.
*   **Blob Storage:** The actual image files are stored in an **object storage system** (like Amazon S3) rather than the database for better performance and scalability.
*   **Cache Tier:** An in-memory cache (like Redis) stores the mapping of `user_id` to `photo_url` to reduce database load during frequent profile views.

### **4. Scale Plan**
*   **Stateless Web Tier:** Use a **Load Balancer** to distribute traffic across multiple API servers, allowing the system to scale horizontally.
*   **Asynchronous Processing:** If features like cropping or thumbnail generation are added, use a **Message Queue** to handle these tasks in the background without blocking the user's upload.
*   **Content Delivery Network (CDN):** If the company has global offices, use a CDN to serve images from the **edge server** closest to the employee.

### **5. Failure Cases**
*   **Cache Inconsistency:** When a user replaces a photo, the system must **invalidate the old cache entry** to ensure the new one is displayed.
*   **Storage Failure:** Use **cross-region replication** for blob storage to guard against data loss if a specific data center goes offline.
*   **Race Conditions:** If two systems update metadata at once, the system should follow a "first-processed wins" strategy to maintain version integrity.

### **6. Security**
*   **Authentication/Authorization:** Integrate with company **Single Sign-On (SSO)** to ensure only the owner (or authorized HR staff) can replace a specific picture.
*   **Pre-signed URLs:** For added security, the server can generate **temporary pre-signed URLs** that give the client permission to upload directly to storage without exposing master credentials.
*   **Input Validation:** Strictly enforce file size limits (e.g., <5MB) and validate image formats to prevent malicious file execution.

### **7. Observability**
*   **SLIs/SLOs:** Monitor the **success rate** of uploads and the **99th percentile latency** for image retrieval to ensure performance targets are met.
*   **Health Checks:** Use independent **probers** to call the service endpoint periodically and verify its health.
*   **Logging:** Centralize error logs to track failed uploads or storage permission issues for rapid debugging.